# Attention Pattern Analysis

Deep analysis of attention patterns: entropy, positional bias, sparsity, cross-head similarity, and automatic classification.

**References:**
- Clark et al. (2019) "What Does BERT Look At?"
- Kovaleva et al. (2019) "Revealing the Dark Secrets of BERT"

## Why This Matters

Systematic attention pattern analysis — entropy, positional bias, sparsity, cross-head similarity, and automatic classification — provides a comprehensive profile of how attention operates across all heads. This is the foundation for identifying specialized heads and understanding information routing.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_pattern_analysis import (
    attention_entropy_profile,
    positional_attention_bias,
    attention_sparsity_analysis,
    cross_head_attention_similarity,
    attention_head_classification,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

In [ ]:
# 1. Attention entropy profile
ent = attention_entropy_profile(model, tokens)
print(f"Max possible entropy: {ent['max_possible_entropy']:.4f}")
print(f"Sharpest head: {ent['sharpest_head']}")
print(f"Most diffuse: {ent['most_diffuse_head']}")
for l in range(cfg.n_layers):
    vals = [f"{ent['entropy_matrix'][l, h]:.3f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {vals} (mean={ent['entropy_by_layer'][l]:.3f})")

In [ ]:
# 2. Positional bias
bias = positional_attention_bias(model, tokens)
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        print(f"L{l}H{h}: BOS={bias['bos_attention'][l,h]:.3f}, "
              f"diag={bias['diagonal_strength'][l,h]:.3f}, "
              f"recency={bias['recency_bias'][l,h]:.3f}, "
              f"mean_dist={bias['mean_attention_distance'][l,h]:.2f}")

In [ ]:
# 3. Sparsity analysis
sp = attention_sparsity_analysis(model, tokens)
print(f"Sparsest: {sp['sparsest_head']}, Densest: {sp['densest_head']}")
for l in range(cfg.n_layers):
    vals = [f"{sp['sparsity_matrix'][l,h]:.3f}" for h in range(cfg.n_heads)]
    gini = [f"{sp['gini_coefficients'][l,h]:.3f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: sparsity={vals}, gini={gini}")

In [ ]:
# 4. Cross-head similarity
sim = cross_head_attention_similarity(model, tokens)
print(f"Most similar pair: {sim['most_similar_pair']}")
print(f"Most dissimilar: {sim['most_dissimilar_pair']}")
print(f"Redundancy score: {sim['redundancy_score']:.4f}")
for l in range(cfg.n_layers):
    print(f"  Layer {l} within-similarity: {sim['within_layer_similarity'][l]:.4f}")

In [ ]:
# 5. Head classification
classif = attention_head_classification(model, tokens)
print(f"Class counts: {classif['class_counts']}")
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        print(f"  L{l}H{h}: {classif['classifications'][l,h]} "
              f"(confidence={classif['confidence_scores'][l,h]:.3f})")